## Imports & Initialization:

In [97]:
%load_ext autoreload
%autoreload 2

import os
import sys
import random
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

from knowledge_maze.tools.utils import load_json, display_usage_log
from knowledge_maze.evaluation.models.choose_models import KnowledgeMazeRunner
from knowledge_maze.evaluation.utils.preprocess import KnowledgeMazeDatasetDistiller
from knowledge_maze.evaluation.models.model_inference import KnowledgeMazeIncrementalRunner
from knowledge_maze.evaluation.models.judge_model import KnowledgeMazeIncrementalJudge

question_data = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_questions.json"))
hedged_question_data = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_questions_hedged.json"))
paragraphs = load_json(os.path.join(REPO_PATH, "data_files/knowledge_maze_paragraphs.json"))

CONFIG_PATH = os.path.join(REPO_PATH, "knowledge_maze/evaluation/models/models_config.yaml")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Preliminary Reliability Assessment

Before conducting the full-scale evaluation of **CLECK**, we perform a preliminary reliability check on each model to ensure it possesses the baseline reasoning capabilities required for the benchmark.

### Methodology

* **Stratified Sampling:** We evaluate each model on a subset of **550 random samples**, stratified across our eleven reasoning operations to prevent logical type bias.
* **Reasoning Verification:** To ensure the model can handle the underlying logic of the task, we execute **three Self-Consistency (SC) rounds** for each reasoning paragraph in isolation.
* **Success Criteria:** A model is considered reliable if it achieves an **overall success rate above 90%** across all test cases.
* **General Success Metric:** Additionally, the model must maintain a **general success rate above 90%**, defined as succeeding in at least **2 out of 3 SC rounds** for any given sample.
* **Validation Goal:** This process confirms that the model’s later failures are due to the **latent conflict** rather than a fundamental inability to perform the multi-hop reasoning or the specific mathematical operations, such as counting or filtering.

---

### The "Diagnostic" Logic

By validating reasoning on isolated hops first, we establish a **capability baseline**. This allows us to argue that when a model defaults to an authoritative claim in the full benchmark, it is a behavioral "lock" rather than a lack of logical skill.

In [136]:
runner = KnowledgeMazeRunner(limit=550)
results_df = await runner.execute(question_data, paragraphs)

Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
deepseek/deepseek-v4-pro,95.2%,97.6%,88.2%,485 52 11 2
mistralai/mistral-medium-3.1,94.4%,94.5%,92.0%,506 14 12 18
meta-llama/llama-4-maverick,94.1%,95.1%,90.4%,497 26 10 17
deepseek/deepseek-v4-flash,94.1%,96.4%,87.6%,482 48 11 9
google/gemini-3-flash-preview,94.1%,94.7%,91.3%,502 19 8 21
openai/gpt-5.4,93.3%,93.5%,91.1%,501 13 10 26
z-ai/glm-4.7-flash,93.2%,94.9%,87.1%,479 43 15 13
openai/gpt-5.4-mini,92.9%,93.3%,88.5%,487 26 20 17


In [135]:
results_df_cleaned = results_df[results_df['Model'] != 'mistralai/mistral-medium-3.1'].copy()
final_df_all = pd.concat([results_df_cleaned, results_df_2, results_df_3, results_df_4])
models = [
    "google/gemini-3-flash-preview",
    "meta-llama/llama-4-maverick",
    "mistralai/mistral-medium-3.1",
    "z-ai/glm-4.7-flash",
    "deepseek/deepseek-v4-flash",
    "openai/gpt-5.4-mini",
    "deepseek/deepseek-v4-pro",
    "openai/gpt-5.4"
]
# 1. Filter the DataFrame to keep only the models in your list
filtered_df = final_df_all[final_df_all['Model'].isin(models)]

# 2. Sort by the success metric in descending order
final_sorted_df = filtered_df.sort_values(by="Mean Success (%)", ascending=False).reset_index(drop=True)

# Display the result
runner.sweep.display_dashboard(final_sorted_df)

Model Architecture,Mean Success,General (≥2/3),Strict (3/3),Stability Profile (3 to 0)
deepseek/deepseek-v4-pro,95.2%,97.6%,88.2%,485 52 11 2
mistralai/mistral-medium-3.1,94.4%,94.5%,92.0%,506 14 12 18
meta-llama/llama-4-maverick,94.1%,95.1%,90.4%,497 26 10 17
deepseek/deepseek-v4-flash,94.1%,96.4%,87.6%,482 48 11 9
google/gemini-3-flash-preview,94.1%,94.7%,91.3%,502 19 8 21
openai/gpt-5.4,93.3%,93.5%,91.1%,501 13 10 26
z-ai/glm-4.7-flash,93.2%,94.9%,87.1%,479 43 15 13
openai/gpt-5.4-mini,92.9%,93.3%,88.5%,487 26 20 17


# 🛠️ Preprocessing Dataset:

### 1. Stratified Population Sampling

To ensure a balanced benchmark, the distiller samples the dataset based on specific research constraints:

* **Hedged Priority**: Automatically includes all **803 Hedged samples** to ensure the uncertainty study is fully represented.
* **Strategy Balancing**: Targets **300 samples per reasoning strategy** (e.g., Temporal Ordering, Multi-hop) for the Full Study.
* **Strict Subsetting**: The **CoT (1.1k)** and **Hedged (803)** datasets are generated as strict subsets of the **Full Study (3.2k)**, allowing for direct paired-sample comparisons.

### 2. Deterministic Context Shuffling

Instead of shuffling paragraphs during the live experiment, the distiller pre-calculates and saves **5 frozen iterations** for every sample:

* **Context IDs**: Each iteration stores a specific list of 10 IDs (1 Evidence, 1 Claim, 8 Distractors) in a fixed sequence.
* **Consistency**: Every model (Gemini, GPT-5, etc.) will see the exact same 5 layouts for Question $X$, ensuring that any performance difference is due to the model architecture, not a "lucky" or "unlucky" shuffle.

### 3. Positional Bias Guardrails

The process includes an automated audit to verify that the shuffling is mathematically "fair":

* **Primacy vs. Recency**: Checks that the Claim does not appear before the Evidence more often than chance (aiming for a ~50/50 split).
* **Edge Salience**: Ensures that both the Truth (Evidence) and the Lie (Claim) occupy high-attention "edge" positions (start/end of context) with equal frequency.

In [89]:
distiller = KnowledgeMazeDatasetDistiller(question_data, paragraphs)
distiller.create_subsets()


════════════════════════════════════════════════════════════
🚀 PREPROCESSING DASHBOARD SUMMARY
════════════════════════════════════════════════════════════
Dataset Name         | Total Samples   |
------------------------------------------------------------
full_study.json      | 2669            |
cot_subset.json      | 1100            |
scaling_subset.json  | 660             |
hedged_subset.json   | 801             |
════════════════════════════════════════════════════════════

### Strategy Distribution Report

| Selected Strategy              | Full           | CoT            | Scaling        | Hedged         |
| ------------------------------ | -------------- | -------------- | -------------- | -------------- |
| AGGREGATION                    | 169 (6.3%)     | 100 (9.1%)     | 60 (9.1%)      | 16 (2.0%)      |
| CATEGORICAL_FILTER_LOOKUP      | 250 (9.4%)     | 100 (9.1%)     | 60 (9.1%)      | 63 (7.9%)      |
| COMPARATIVE_DIFFERENCE         | 250 (9.4%)     | 100 (9.1%)     | 

# 🚀 Central Execution Logic: The 4 Experimental Tiers

The framework isolates model behavior by running four distinct experimental iterations using a centralized configuration file. The structure ensures strict control variables by standardizing the output schema (`Explanation:` + ``) across all runs, separating reasoning capacity from vocabulary limits.

---

### 1. 📊 Baseline Run ($n=2,600$)
* **Core Objective:** Establish the standard performance floor for the 5 selected Flash/Open models under a uniform, unguided scenario.
* **Prompt Gating:** Uses `BASELINE_PROMPT` (forces an explanation block, but provides zero explicit reasoning steps).
* **Data Context:** Standard, unhedged source paragraphs (`full_study.json`).

### 2. 🧠 Chain-of-Thought (CoT) Ablation ($n=1,100$)
* **Core Objective:** Measure how explicit, step-by-step reasoning guidance shifts conflict awareness and final source preference.
* **Prompt Gating:** Uses `COT_PROMPT` (explicitly forces the model to catalog all relevant text pieces and reason through them sequentially before answering).
* **Data Context:** Core subset evaluation (`cot_subset.json`).

### 3. 📈 Scaling Analysis ($n=660$)
* **Core Objective:** Evaluate whether conflict awareness is an emergent property of model parameter scale by matching a distilled "mini/flash" tier directly against its flagship equivalent.
* **Prompt Gating:** Uses `BASELINE_PROMPT` to isolate raw scaling capacity from engineered prompt guidance.
* **Data Context:** Focused tier comparison subset (`scaling_subset.json`) evaluating `gpt-5-mini` vs. `gpt-5`, and `deepseek-v4-flash` vs. `deepseek-v4-pro`.

### 4. 🌫️ Hedged Context Ablation ($n=800$)
* **Core Objective:** Assess how a model handles explicit, linguistic uncertainty (hedging terms like *“allegedly,” “potentially,”* or *“some sources claim”*) embedded directly within the evidence.
* **Prompt Gating:** Uses `BASELINE_PROMPT` to evaluate if raw linguistic uncertainty forces spontaneous conflict awareness.
* **Data Context:** The runner automatically activates the `use_hedged_text = True` logic, swapping standard target paragraphs for their uncertainty-modified variants (`hedged_subset.json`).

In [109]:
baseline_runner = KnowledgeMazeIncrementalRunner(config_path=CONFIG_PATH, experiment_type="baseline")
await baseline_runner.run_incremental(sample_limit=80)


>>> Model: google/gemini-3-flash-preview | Mode: STANDARD
--- Found 80 pending samples. Generating frozen prompts...


Inference: google/gemini-3-flash-preview:   0%|          | 0/400 [00:00<?, ?it/s]

>>> Completed run for google/gemini-3-flash-preview.

>>> Model: openai/gpt-5-mini | Mode: STANDARD
--- Found 80 pending samples. Generating frozen prompts...


Inference: openai/gpt-5-mini:   0%|          | 0/400 [00:00<?, ?it/s]

>>> Completed run for openai/gpt-5-mini.

>>> Model: meta-llama/llama-4-maverick | Mode: STANDARD
--- Found 80 pending samples. Generating frozen prompts...


...

>>> Completed run for meta-llama/llama-4-maverick.

>>> Model: deepseek/deepseek-v4-flash | Mode: STANDARD
--- Found 80 pending samples. Generating frozen prompts...


Inference: deepseek/deepseek-v4-flash:   0%|          | 0/400 [00:00<?, ?it/s]

>>> Completed run for deepseek/deepseek-v4-flash.

>>> Model: z-ai/glm-4.7-flash | Mode: STANDARD
--- Found 80 pending samples. Generating frozen prompts...


Inference: z-ai/glm-4.7-flash:   0%|          | 0/400 [00:00<?, ?it/s]

API Error: 'NoneType' object is not subscriptable
>>> Completed run for z-ai/glm-4.7-flash.


In [93]:
cot_runner = KnowledgeMazeIncrementalRunner(config_path=CONFIG_PATH, experiment_type="cot")
await cot_runner.run_incremental(sample_limit=20)


>>> Model: google/gemini-3-flash-preview | Mode: STANDARD
--- Found 20 pending samples. Generating frozen prompts...


Inference: google/gemini-3-flash-preview:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for google/gemini-3-flash-preview.

>>> Model: meta-llama/llama-4-maverick | Mode: STANDARD
--- Found 20 pending samples. Generating frozen prompts...


Inference: meta-llama/llama-4-maverick:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for meta-llama/llama-4-maverick.

>>> Model: deepseek/deepseek-v4-flash | Mode: STANDARD
--- Found 20 pending samples. Generating frozen prompts...


Inference: deepseek/deepseek-v4-flash:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for deepseek/deepseek-v4-flash.


In [94]:
hedge_runner = KnowledgeMazeIncrementalRunner(config_path=CONFIG_PATH, experiment_type="hedged")
await hedge_runner.run_incremental(sample_limit=20)


>>> Model: google/gemini-3-flash-preview | Mode: HEDGED
--- Found 20 pending samples. Generating frozen prompts...


Inference: google/gemini-3-flash-preview:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for google/gemini-3-flash-preview.

>>> Model: meta-llama/llama-4-maverick | Mode: HEDGED
--- Found 20 pending samples. Generating frozen prompts...


Inference: meta-llama/llama-4-maverick:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for meta-llama/llama-4-maverick.

>>> Model: deepseek/deepseek-v4-flash | Mode: HEDGED
--- Found 20 pending samples. Generating frozen prompts...


Inference: deepseek/deepseek-v4-flash:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for deepseek/deepseek-v4-flash.


In [95]:
scaling_runner = KnowledgeMazeIncrementalRunner(config_path=CONFIG_PATH, experiment_type="scaling")
await scaling_runner.run_incremental(sample_limit=20)


>>> Model: deepseek/deepseek-v4-pro | Mode: STANDARD
--- Found 20 pending samples. Generating frozen prompts...


Inference: deepseek/deepseek-v4-pro:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for deepseek/deepseek-v4-pro.

>>> Model: openai/gpt-5 | Mode: STANDARD
--- Found 20 pending samples. Generating frozen prompts...


Inference: openai/gpt-5:   0%|          | 0/100 [00:00<?, ?it/s]

>>> Completed run for openai/gpt-5.


# Asses Ans with Judge

In [123]:
evaluator_baseline = KnowledgeMazeIncrementalJudge(config_path=CONFIG_PATH, experiment_type="baseline")
await evaluator_baseline.run_judge_incremental()


⚖️ Judging model output folder: BASELINE | File: deepseek_deepseek-v4-flash.json
--- Processing all 80 pending entries.


Evaluating: deepseek_deepseek-v4-flash.json:   0%|          | 0/400 [00:00<?, ?it/s]

✅ State updated and committed for deepseek_deepseek-v4-flash.json.

⚖️ Judging model output folder: BASELINE | File: google_gemini-3-flash-preview.json
--- Processing all 80 pending entries.


Evaluating: google_gemini-3-flash-preview.json:   0%|          | 0/400 [00:00<?, ?it/s]

API Error: 'NoneType' object is not subscriptable
API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771700000'}, 'provider_name': None}}, 'user_id': 'user_35Skf82uXTS5c3YrGdp1ojouqwk'}
API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771700000'}, 'provider_name':

...

API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771700000'}, 'provider_name': None}}, 'user_id': 'user_35Skf82uXTS5c3YrGdp1ojouqwk'}
API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771700000'}, 'provider_name': None}}, 'user_id': 'user_35Skf82uXTS5c3YrGdp1ojou

Evaluating: openai_gpt-5-mini.json:   0%|          | 0/400 [00:00<?, ?it/s]

API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771760000'}, 'provider_name': None}}, 'user_id': 'user_35Skf82uXTS5c3YrGdp1ojouqwk'}
API Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/google/gemini-3.1-flash-lite-preview-20260303/7bed1a52-857b-4437-a691-0064e082b1d5. High demand for google/gemini-3.1-flash-lite-preview on OpenRouter - limited to 300 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '300', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1778771760000'}, 'provider_name': None}}, 'user_id': 'user_35Skf82uXTS5c3YrGdp1ojou

Evaluating: z-ai_glm-4.7-flash.json:   0%|          | 0/400 [00:00<?, ?it/s]

✅ State updated and committed for z-ai_glm-4.7-flash.json.


In [119]:
evaluator_cot = KnowledgeMazeIncrementalJudge(config_path=CONFIG_PATH, experiment_type="cot")
await evaluator_cot.run_judge_incremental()


⚖️ Judging model output folder: COT | File: deepseek_deepseek-v4-flash.json
--- Processing all 20 pending entries.


Evaluating: deepseek_deepseek-v4-flash.json:   0%|          | 0/100 [00:00<?, ?it/s]

✅ State updated and committed for deepseek_deepseek-v4-flash.json.

⚖️ Judging model output folder: COT | File: google_gemini-3-flash-preview.json
--- Processing all 20 pending entries.


Evaluating: google_gemini-3-flash-preview.json:   0%|          | 0/100 [00:00<?, ?it/s]

✅ State updated and committed for google_gemini-3-flash-preview.json.

⚖️ Judging model output folder: COT | File: meta-llama_llama-4-maverick.json
--- Processing all 20 pending entries.


Evaluating: meta-llama_llama-4-maverick.json:   0%|          | 0/100 [00:00<?, ?it/s]

✅ State updated and committed for meta-llama_llama-4-maverick.json.


# Show results

In [124]:
deepseek_baseline_res = load_json(os.path.join(REPO_PATH, "results/baseline/deepseek_deepseek-v4-flash.json"))
google_baseline_res = load_json(os.path.join(REPO_PATH, "results/baseline/google_gemini-3-flash-preview.json"))
meta_baseline_res = load_json(os.path.join(REPO_PATH, "results/baseline/meta-llama_llama-4-maverick.json"))
gpt_baseline_res = load_json(os.path.join(REPO_PATH, "results/baseline/openai_gpt-5-mini.json"))
glm_baseline_res = load_json(os.path.join(REPO_PATH, "results/baseline/z-ai_glm-4.7-flash.json"))

In [120]:
deepseek_cot_res = load_json(os.path.join(REPO_PATH, "results/cot/deepseek_deepseek-v4-flash.json"))
google_cot_res = load_json(os.path.join(REPO_PATH, "results/cot/google_gemini-3-flash-preview.json"))
meta_cot_res = load_json(os.path.join(REPO_PATH, "results/cot/meta-llama_llama-4-maverick.json"))

In [121]:
import numpy as np
from collections import Counter
def get_metric(res):
    ack_rate = []
    class_rate = []
    for s in list(res.values()):
        sample_ack = []
        sample_class = []
        for a in s["responses"]:
            if "judge_results" in a.keys():
                sample_ack.append(a["judge_results"]["conflict_acknowledged"])
                sample_class.append(a["judge_results"]["classification"])
        ack_rate.append(np.mean(sample_ack))
    cleaned_ack_rate = [x for x in ack_rate if not np.isnan(x)]
        
    print(Counter(cleaned_ack_rate))
    # print(Counter(class_rate))

In [125]:
get_metric(deepseek_baseline_res)
get_metric(google_baseline_res)
get_metric(meta_baseline_res)
get_metric(gpt_baseline_res)
get_metric(glm_baseline_res)

Counter({0.0: 82, 0.2: 14, 0.4: 4})
Counter({0.0: 47, 0.2: 16, 0.4: 14, 0.8: 9, 1.0: 8, 0.6: 6})
Counter({0.0: 54, 0.2: 20, 0.4: 16, 0.6: 7, 0.8: 2, 1.0: 1})
Counter({0.0: 60, 0.2: 23, 0.4: 11, 0.6: 4, 0.8: 1, 1.0: 1})
Counter({0.0: 73, 0.2: 23, 0.4: 4})


In [122]:
get_metric(deepseek_cot_res)
get_metric(google_cot_res)
get_metric(meta_cot_res)

Counter({0.0: 6, 0.6: 5, 0.4: 5, 0.8: 2, 0.2: 2})
Counter({1.0: 7, 0.0: 7, 0.2: 2, 0.8: 2, 0.6: 1, 0.4: 1})
Counter({0.0: 7, 0.4: 5, 0.6: 3, 0.2: 3, 0.8: 1, 1.0: 1})
